In [1]:
import duckdb
import pandas as pd
from pathlib import Path

print("DuckDB:", duckdb.__version__)

DuckDB: 1.5.5


In [3]:
from pathlib import Path

ROOT = Path(r"C:\Users\bbert_xwhb76k\analises-GIS")
PROJECT = ROOT / "01-monitoramento-vegetacao"
RESULTS = PROJECT / "results"
PARQUET_DIR = PROJECT / "data" / "processed" / "parquet"

PARQUET_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(RESULTS.rglob("*.csv"))

print(f"CSV encontrados: {len(csv_files)}")
print()

for f in csv_files:
    print(f.relative_to(PROJECT))

CSV encontrados: 28

results\tables\anomalias_por_data.csv
results\tables\correlacao_2020_indices.csv
results\tables\correlacao_2020_spearman.csv
results\tables\correlacao_2021_indices.csv
results\tables\correlacao_2021_spearman.csv
results\tables\correlacao_2022_indices.csv
results\tables\correlacao_2022_spearman.csv
results\tables\correlacao_2023_indices.csv
results\tables\correlacao_2023_spearman.csv
results\tables\correlacao_2024_indices.csv
results\tables\correlacao_2024_spearman.csv
results\tables\correlacao_2025_indices.csv
results\tables\correlacao_2025_spearman.csv
results\tables\correlacao_indices.csv
results\tables\estatisticas_espaciais.csv
results\tables\estatisticas_espaciais_2020.csv
results\tables\estatisticas_espaciais_2021.csv
results\tables\estatisticas_espaciais_2022.csv
results\tables\estatisticas_espaciais_2023.csv
results\tables\estatisticas_espaciais_2024.csv
results\tables\estatisticas_espaciais_2025.csv
results\tables\hotspots_anomalia.csv
results\tables\sazon

In [4]:
import duckdb

conn = duckdb.connect()

principais = [
    "serie_anual_completa.csv",
    "serie_intraanual_completa.csv",
    "anomalias_por_data.csv",
    "hotspots_anomalia.csv",
    "sazonalidade.csv",
    "estatisticas_espaciais.csv",
]

for nome in principais:
    arquivo = RESULTS / "tables" / nome

    print("\n" + "=" * 80)
    print(nome)
    print("=" * 80)

    df_info = conn.execute(f"""
        DESCRIBE
        SELECT *
        FROM read_csv_auto('{arquivo.as_posix()}')
    """).df()

    print(df_info.to_string(index=False))


serie_anual_completa.csv
column_name column_type null  key default extra
     indice     VARCHAR  YES None    None  None
        ano      BIGINT  YES None    None  None
      media      DOUBLE  YES None    None  None
    mediana      DOUBLE  YES None    None  None
        std      DOUBLE  YES None    None  None
        min      DOUBLE  YES None    None  None
        max      DOUBLE  YES None    None  None
    n_datas      BIGINT  YES None    None  None

serie_intraanual_completa.csv
column_name column_type null  key default extra
       data        DATE  YES None    None  None
   data_str      BIGINT  YES None    None  None
        ano      BIGINT  YES None    None  None
        mes      BIGINT  YES None    None  None
        dia      BIGINT  YES None    None  None
     indice     VARCHAR  YES None    None  None
      media      DOUBLE  YES None    None  None
    mediana      DOUBLE  YES None    None  None
        std      DOUBLE  YES None    None  None
        p25      DOUBLE  YES No

In [5]:
from pathlib import Path
import duckdb

TABLES = RESULTS / "tables"
PARQUET_DIR = PROJECT / "data" / "processed" / "parquet"

PARQUET_DIR.mkdir(parents=True, exist_ok=True)

conn = duckdb.connect()

# ---------------------------------------------------------
# 1. Série anual
# ---------------------------------------------------------

conn.execute(f"""
COPY (
    SELECT
        indice,
        CAST(ano AS INTEGER) AS ano,
        media,
        mediana,
        std,
        min,
        max,
        CAST(n_datas AS INTEGER) AS n_datas
    FROM read_csv_auto('{(TABLES / "serie_anual_completa.csv").as_posix()}')
)
TO '{(PARQUET_DIR / "serie_anual.parquet").as_posix()}'
(FORMAT PARQUET);
""")


# ---------------------------------------------------------
# 2. Série intra-anual
# ---------------------------------------------------------

conn.execute(f"""
COPY (
    SELECT
        data,
        CAST(ano AS INTEGER) AS ano,
        CAST(mes AS INTEGER) AS mes,
        CAST(dia AS INTEGER) AS dia,
        indice,
        media,
        mediana,
        std,
        p25,
        p75,
        min,
        max,
        CAST(n_pixels AS INTEGER) AS n_pixels
    FROM read_csv_auto('{(TABLES / "serie_intraanual_completa.csv").as_posix()}')
)
TO '{(PARQUET_DIR / "serie_intraanual.parquet").as_posix()}'
(FORMAT PARQUET);
""")


# ---------------------------------------------------------
# 3. Anomalias
# ---------------------------------------------------------

conn.execute(f"""
COPY (
    SELECT
        indice,
        data,
        CAST(ano AS INTEGER) AS ano,
        CAST(mes AS INTEGER) AS mes,
        media,
        mediana,
        std,
        CAST(n_pixels AS INTEGER) AS n_pixels,
        media_sazonal,
        std_sazonal,
        CAST(n AS INTEGER) AS n,
        anomalia,
        z_score,
        classificacao
    FROM read_csv_auto('{(TABLES / "anomalias_por_data.csv").as_posix()}')
)
TO '{(PARQUET_DIR / "anomalias.parquet").as_posix()}'
(FORMAT PARQUET);
""")


# ---------------------------------------------------------
# 4. Hotspots
# ---------------------------------------------------------

conn.execute(f"""
COPY (
    SELECT
        indice,
        tipo,
        CAST(id_cluster AS INTEGER) AS id_cluster,
        CAST(n_pixels AS INTEGER) AS n_pixels,
        area_ha,
        limiar_persistencia_pct,
        limiar_anomalia
    FROM read_csv_auto('{(TABLES / "hotspots_anomalia.csv").as_posix()}')
)
TO '{(PARQUET_DIR / "hotspots.parquet").as_posix()}'
(FORMAT PARQUET);
""")


# ---------------------------------------------------------
# 5. Sazonalidade
# ---------------------------------------------------------

conn.execute(f"""
COPY (
    SELECT
        indice,
        CAST(mes AS INTEGER) AS mes,
        media,
        mediana,
        std
    FROM read_csv_auto('{(TABLES / "sazonalidade.csv").as_posix()}')
)
TO '{(PARQUET_DIR / "sazonalidade.parquet").as_posix()}'
(FORMAT PARQUET);
""")


# ---------------------------------------------------------
# 6. Estatísticas espaciais
# ---------------------------------------------------------

conn.execute(f"""
COPY (
    SELECT
        "Índice" AS indice,
        CAST("N válidos" AS INTEGER) AS n_validos,
        CAST("N NaN" AS INTEGER) AS n_nan,
        "Mín" AS minimo,
        P25 AS p25,
        Mediana AS mediana,
        Média AS media,
        P75 AS p75,
        "Máx" AS maximo,
        "Desv Pad" AS desvio_padrao
    FROM read_csv_auto('{(TABLES / "estatisticas_espaciais.csv").as_posix()}')
)
TO '{(PARQUET_DIR / "estatisticas_espaciais.parquet").as_posix()}'
(FORMAT PARQUET);
""")

print("Conversão concluída.")

Conversão concluída.


In [6]:
parquets = sorted(PARQUET_DIR.glob("*.parquet"))

print(f"Parquets criados: {len(parquets)}\n")

for arquivo in parquets:
    info = conn.execute(f"""
        SELECT
            COUNT(*) AS linhas
        FROM read_parquet('{arquivo.as_posix()}')
    """).fetchone()

    print(f"{arquivo.name:<35} {info[0]:>8} linhas")

Parquets criados: 6

anomalias.parquet                        207 linhas
estatisticas_espaciais.parquet             3 linhas
hotspots.parquet                          35 linhas
sazonalidade.parquet                      36 linhas
serie_anual.parquet                       18 linhas
serie_intraanual.parquet                 207 linhas


In [7]:
for arquivo in sorted(PARQUET_DIR.glob("*.parquet")):
    print("\n" + "=" * 80)
    print(arquivo.name)
    print("=" * 80)

    resultado = conn.execute(f"""
        DESCRIBE
        SELECT *
        FROM read_parquet('{arquivo.as_posix()}')
    """).df()

    print(resultado.to_string(index=False))


anomalias.parquet
  column_name column_type null  key default extra
       indice     VARCHAR  YES None    None  None
         data        DATE  YES None    None  None
          ano     INTEGER  YES None    None  None
          mes     INTEGER  YES None    None  None
        media      DOUBLE  YES None    None  None
      mediana      DOUBLE  YES None    None  None
          std      DOUBLE  YES None    None  None
     n_pixels     INTEGER  YES None    None  None
media_sazonal      DOUBLE  YES None    None  None
  std_sazonal      DOUBLE  YES None    None  None
            n     INTEGER  YES None    None  None
     anomalia      DOUBLE  YES None    None  None
      z_score      DOUBLE  YES None    None  None
classificacao     VARCHAR  YES None    None  None

estatisticas_espaciais.parquet
  column_name column_type null  key default extra
       indice     VARCHAR  YES None    None  None
    n_validos     INTEGER  YES None    None  None
        n_nan     INTEGER  YES None    None  None

In [8]:
for arquivo in sorted(PARQUET_DIR.glob("*.parquet")):
    print("\n" + "=" * 80)
    print(arquivo.name)
    print("=" * 80)

    df = conn.execute(f"""
        SELECT *
        FROM read_parquet('{arquivo.as_posix()}')
        LIMIT 5
    """).df()

    display(df)


anomalias.parquet


,indice,data,ano,mes,media,mediana,std,n_pixels,media_sazonal,std_sazonal,n,anomalia,z_score,classificacao
0,NDVI,2020-01-15,2020,1,0.309165,0.263145,0.293134,31080,0.414228,0.079520,5,-0.105063,-1.321209,Abaixo
1,NDVI,2020-02-14,2020,2,0.503374,0.766316,0.402265,31080,0.486078,0.057024,5,0.017296,0.303312,Normal
2,NDVI,2020-03-25,2020,3,0.522443,0.820190,0.413787,31080,0.390431,0.083181,6,0.132012,1.587039,Muito acima
3,NDVI,2020-04-04,2020,4,0.461980,0.554771,0.380069,31080,0.337883,0.062709,6,0.124097,1.978935,Muito acima
4,NDVI,2020-05-19,2020,5,0.300030,0.224212,0.304312,31080,0.331926,0.027087,6,-0.031896,-1.177514,Abaixo



estatisticas_espaciais.parquet


,indice,n_validos,n_nan,minimo,p25,mediana,media,p75,maximo,desvio_padrao
0,NDVI,19358,11722,0.130039,0.375199,0.489448,0.541631,0.749393,0.899400,0.203387
1,EVI,31080,0,0.000000,0.000000,0.219268,0.201238,0.336986,0.681808,0.184012
2,NDMI,19780,11300,-0.280027,-0.137886,-0.025009,0.020361,0.181200,0.464676,0.189813



hotspots.parquet


...

,indice,mes,media,mediana,std
0,EVI,1,0.301545,0.385798,0.066381
1,EVI,2,0.403120,0.487031,0.111616
2,EVI,3,0.328998,0.333388,0.100545
3,EVI,4,0.209198,0.147345,0.049417
4,EVI,5,0.208365,0.181235,0.046967



serie_anual.parquet


,indice,ano,media,mediana,std,min,max,n_datas
0,EVI,2020,0.283348,0.192562,0.200283,0.000000,1.440746,12
1,EVI,2021,0.231034,0.214265,0.189550,0.000000,1.755039,12
2,EVI,2022,0.323643,0.383929,0.184404,-1.133468,2.820700,12
3,EVI,2023,0.250805,0.226347,0.202757,0.000000,0.932064,11
4,EVI,2024,0.256408,0.263644,0.176956,0.000000,1.543158,10



serie_intraanual.parquet


,data,ano,mes,dia,indice,media,mediana,std,p25,p75,min,max,n_pixels
0,2020-01-15,2020,1,15,EVI,0.223009,0.196038,0.216389,0.0,0.396003,0.0,0.814994,31080
1,2020-02-14,2020,2,14,EVI,0.366175,0.450718,0.316177,0.0,0.626960,0.0,1.018524,31080
2,2020-03-25,2020,3,25,EVI,0.380476,0.496871,0.315150,0.0,0.636567,0.0,0.891410,31080
3,2020-04-04,2020,4,4,EVI,0.307858,0.379992,0.256715,0.0,0.536238,0.0,0.841613,31080
4,2020-05-19,2020,5,19,EVI,0.155751,0.124041,0.164454,0.0,0.249630,0.0,0.696395,31080


In [9]:
from pathlib import Path

PROJECT = Path(r"C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao")

for path in sorted(PROJECT.rglob("*")):
    if path.is_dir():
        nivel = len(path.relative_to(PROJECT).parts)
        if nivel <= 3:
            print("    " * (nivel - 1) + f"📁 {path.name}/")
    else:
        nivel = len(path.relative_to(PROJECT).parts)
        if nivel <= 3:
            print("    " * (nivel - 1) + f"📄 {path.name}")

📁 data/
    📄 .gitkeep
    📁 processed/
        📄 .gitkeep
        📁 .ipynb_checkpoints/
        📁 indices/
        📁 parquet/
    📁 raw/
        📄 .gitkeep
        📁 Area_de_Preservacao_Permanente/
        📄 Area_de_Preservacao_Permanente.zip
        📁 Area_de_Uso_Restrito/
        📄 Area_de_Uso_Restrito.zip
        📁 Area_do_Imovel/
        📄 Area_do_Imovel.zip
        📁 Cobertura_do_Solo/
        📄 Cobertura_do_Solo.zip
        📁 MARCADORES_Area_de_Preservacao_Permanente/
        📄 MARCADORES_Area_de_Preservacao_Permanente.zip
        📄 propriedade.geojson
        📁 Reserva_Legal/
        📄 Reserva_Legal.zip
    📁 reference/
        📄 .gitkeep
📁 docs/
    📄 metodologia.pdf
📁 geoserver/
    📄 .gitkeep
📁 notebooks/
    📁 .ipynb_checkpoints/
        📄 09_analise_temporal_indices-Copy1-checkpoint.ipynb
        📄 10_analise_espacial_indices-checkpoint.ipynb
        📄 11_analise_anomalias_indices-checkpoint.ipynb
        📄 Untitled-checkpoint.ipynb
    📄 01_preparacao_projeto.ipynb
    📄 

In [10]:
from pathlib import Path
import shutil

PROJECT = Path(r"C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao")

# ---------------------------------------------------------
# 1. Criar novas pastas
# ---------------------------------------------------------

for pasta in [
    PROJECT / "analytics",
    PROJECT / "dashboard",
    PROJECT / "gis",
]:
    pasta.mkdir(exist_ok=True)


# ---------------------------------------------------------
# 2. Mover web/dashboard → dashboard
# ---------------------------------------------------------

origem_dashboard = PROJECT / "web" / "dashboard"
destino_dashboard = PROJECT / "dashboard"

if origem_dashboard.exists():

    # Se dashboard estiver vazio, podemos mover o conteúdo
    for item in origem_dashboard.iterdir():

        destino = destino_dashboard / item.name

        if destino.exists():
            print(f"⚠️ Já existe, não movido: {destino}")
        else:
            shutil.move(str(item), str(destino))
            print(f"Movido: {item.relative_to(PROJECT)} → {destino.relative_to(PROJECT)}")

    # Remove web se ficou vazio
    try:
        origem_dashboard.rmdir()
        (PROJECT / "web").rmdir()
        print("Removido: web/")
    except OSError:
        print("ℹ️ web/ não estava vazio; mantido.")


# ---------------------------------------------------------
# 3. Mover componentes GIS
# ---------------------------------------------------------

for nome in ["qgis", "postgis", "geoserver"]:

    origem = PROJECT / nome
    destino = PROJECT / "gis" / nome

    if origem.exists():

        if destino.exists():
            print(f"⚠️ Já existe: {destino}")
        else:
            shutil.move(str(origem), str(destino))
            print(f"Movido: {nome}/ → gis/{nome}/")


# ---------------------------------------------------------
# 4. Criar arquivos __init__.py
# ---------------------------------------------------------

for pasta in [
    PROJECT / "analytics",
    PROJECT / "dashboard",
]:
    init_file = pasta / "__init__.py"

    if not init_file.exists():
        init_file.touch()
        print(f"Criado: {init_file.relative_to(PROJECT)}")


print("\n✅ Reorganização concluída.")

Removido: web/
Movido: qgis/ → gis/qgis/
Movido: postgis/ → gis/postgis/
Movido: geoserver/ → gis/geoserver/
Criado: analytics\__init__.py
Criado: dashboard\__init__.py

✅ Reorganização concluída.
